# mT5 disperso: upcycling a Mixture of Experts

Convierte `google/mt5-small` en un modelo MoE y lo ajusta para normalizar chileno → español neutro. Usa los mismos datos, partición e hiperparámetros que `models/mt5.ipynb`.

**Ruta:** `google/mt5-small` → upcycling a MoE → entrenamiento → evaluación.

**Antes de ejecutar:** en Colab, elige *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* y luego *Ejecutar todas*. Colab pedirá autorizar Google Drive.

**Se guarda en Google Drive:**

| Contenido | Ruta |
|---|---|
| Pesos, configuración y tokenizer | `/content/drive/MyDrive/proyecto-nlp/models/mt5_moe_fp32/` |
| Predicciones | `/content/drive/MyDrive/proyecto-nlp/data/processed/predicciones_mt5_moe_fp32.csv` |
| BLEU y chrF, junto al denso | `/content/drive/MyDrive/proyecto-nlp/resultados/comparacion_mt5.csv` |

Los pesos ocupan cerca de 1,3 GB. El mejor modelo queda en el disco de Colab durante el entrenamiento y se copia a Drive una sola vez al terminar.

## Arquitectura

| | mT5 denso | mT5 MoE |
|---|---|---|
| `d_model`, `d_ff` | 512, 1024 | igual |
| Bloques por lado | 8 | igual |
| Capa feed-forward | gated-GELU (`wi_0`, `wi_1`, `wo`) | igual en cada experto |
| Capas dispersas | ninguna | bloques 1, 3, 5 y 7 de encoder y decoder |
| Expertos por capa dispersa | – | 4, y cada token usa los 2 mejores |
| Pérdidas auxiliares | – | balance de carga y z-loss, 0,001 cada una |
| Ruido del enrutador | – | 0,01 |
| Optimización | Adafactor, lr 5e-4, 8 épocas, lote 8, semilla 1, fp32 | igual |

**Qué se toma de Switch Transformers:** el enrutador lineal por token, el ruido multiplicativo, la selección de capas dispersas y las dos pérdidas auxiliares (Fedus et al., 2021; Fedus et al., 2022).

**Qué cambia:**
- No se usa `SwitchTransformersForConditionalGeneration`, porque enruta a un solo experto y su checkpoint es T5 en inglés con otro vocabulario.
- Se enruta a 2 expertos y sus pesos se renormalizan para sumar 1. Así, al copiar el mismo experto cuatro veces, el modelo recién convertido calcula exactamente lo mismo que el denso (*sparse upcycling*, Komatsuzaki et al., 2022).
- El ruido solo afecta la entrada del enrutador, no la de los expertos.
- No hay límite de capacidad por experto, para que ningún token quede sin procesar.
- No se usa *gradient checkpointing*: la T4 tiene memoria suficiente y así las pérdidas del enrutador conservan su gradiente.

## 1. Configuración

In [ ]:
RAMA = "entrega-final"
URL_REPO = "https://github.com/carlosiskindacool/proyecto-nlp.git"
MODELO = "google/mt5-small"

# Igual que el mT5 denso
PRECISION = "fp32"  # "fp32" o "bf16". La T4 no tiene bf16 por hardware: usar fp32.
EPOCAS = 8
LOTE = 8
LR = 5e-4
VAL = 0.05
SEMILLA = 1
MAX_LENGTH = 128

# Parámetros del MoE
NUM_EXPERTOS = 4
TOP_K = 2
PASO_DISPERSO = 2  # una capa de cada dos: bloques 1, 3, 5 y 7
COEF_BALANCE = 0.001
COEF_Z = 0.001
RUIDO_ROUTER = 0.01

SOBRESCRIBIR = False  # True para reemplazar pesos ya guardados en Drive

ENTRENAMIENTO = "data/interim/borrador_entrenamiento.csv"
PARES = ["annotation/lote1.csv", "annotation/lote_1_noticias.csv", "annotation/lote_1_tuits.csv"]

if PRECISION not in ("bf16", "fp32"):
    raise ValueError('PRECISION debe ser "bf16" o "fp32"')

NOMBRE = f"mt5_moe_{PRECISION}"
RAIZ_DRIVE = "/content/drive"
CARPETA_DRIVE = f"{RAIZ_DRIVE}/MyDrive/proyecto-nlp"
DIR_PESOS = f"{CARPETA_DRIVE}/models/{NOMBRE}"
RUTA_PREDICCIONES = f"{CARPETA_DRIVE}/data/processed/predicciones_{NOMBRE}.csv"
RUTA_RESUMEN = f"{CARPETA_DRIVE}/resultados/comparacion_mt5.csv"
DIR_LOCAL = f"/content/{NOMBRE}_mejor"
REPO = "/content/proyecto-nlp-moe"

## 2. Montar Google Drive

In [ ]:
import os
import shutil

from google.colab import drive

drive.mount(RAIZ_DRIVE)
if not os.path.isdir(f"{RAIZ_DRIVE}/MyDrive"):
    raise RuntimeError("Google Drive no quedó montado; los pesos se perderían al cerrar Colab.")

if os.path.exists(f"{DIR_PESOS}/model.safetensors") and not SOBRESCRIBIR:
    raise FileExistsError(f"Ya hay pesos en {DIR_PESOS}. Usa SOBRESCRIBIR = True o cambia NOMBRE.")

for carpeta in (os.path.dirname(DIR_PESOS), os.path.dirname(RUTA_PREDICCIONES), os.path.dirname(RUTA_RESUMEN)):
    os.makedirs(carpeta, exist_ok=True)

libre_gb = shutil.disk_usage(f"{RAIZ_DRIVE}/MyDrive").free / 1024**3
print(f"Drive montado | libre: {libre_gb:.1f} GB | pesos en: {DIR_PESOS}")
if libre_gb < 3:
    print("Aviso: conviene tener al menos 3 GB libres en Drive para copiar los pesos.")

## 3. Instalar dependencias y traer el repositorio

In [ ]:
!pip install -q "transformers==5.24.0" sentencepiece protobuf sacrebleu

In [ ]:
import subprocess
import sys

if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, URL_REPO, REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src/chilenismos")

## 4. Preparar GPU, precisión y datos

In [ ]:
import csv
import json
import math
import random

import sacrebleu
import torch
from transformers import Adafactor, AutoModelForSeq2SeqLM, AutoTokenizer

from baselines import cargar_pares, guardar_resumen, limpiar

random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev != "cuda":
    raise RuntimeError("No hay GPU. Activa la T4 en Entorno de ejecución → Cambiar tipo de entorno.")

USAR_BF16 = PRECISION == "bf16" and dev == "cuda"
print("dispositivo:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "", "| precisión:", PRECISION)

In [ ]:
def cargar_entrenamiento(ruta, prohibidas):
    with open(ruta, encoding="utf-8") as f:
        filas = list(csv.DictReader(f, delimiter=";"))
    vistas, pares = set(), []
    for r in filas:
        o, n = limpiar(r["oracion"]), limpiar(r["neutro"])  # sin menciones ni emojis, igual que los conjuntos de prueba
        if not o or not n or n.upper() == "DESCARTAR" or o in prohibidas or o in vistas:
            continue
        vistas.add(o)
        pares.append((o, n))
    return pares


conjuntos = {ruta: cargar_pares(ruta) for ruta in PARES}
prohibidas = {p["chileno"] for ps in conjuntos.values() for p in ps}  # evita fuga hacia la evaluación
pares = cargar_entrenamiento(ENTRENAMIENTO, prohibidas)
random.shuffle(pares)
n_val = max(1, round(len(pares) * VAL))
val, train = pares[:n_val], pares[n_val:]
print(f"entrenamiento: {len(train)} pares | validación: {len(val)} | prueba: {sum(map(len, conjuntos.values()))}")

## 5. Definir el modelo MoE

In [ ]:
from huggingface_hub.dataclasses import strict
from torch import nn
from torch.nn import functional as F
from transformers import MT5Config, MT5ForConditionalGeneration
from transformers.models.mt5.modeling_mt5 import MT5DenseGatedActDense


@strict
class MT5MoEConfig(MT5Config):
    model_type = "mt5_moe"
    moe_expertos: int = 4
    moe_top_k: int = 2
    moe_paso_disperso: int = 2
    moe_ruido_router: float = 0.01


class CapaMoE(nn.Module):
    """Reemplaza la feed-forward densa: cada token pasa por sus `top_k` expertos."""

    def __init__(self, config, es_decoder):
        super().__init__()
        self.num_expertos = config.moe_expertos
        self.top_k = config.moe_top_k
        self.ruido = config.moe_ruido_router
        self.es_decoder = es_decoder
        self.router = nn.Linear(config.d_model, config.moe_expertos, bias=False)
        self.expertos = nn.ModuleList(MT5DenseGatedActDense(config) for _ in range(config.moe_expertos))
        self.registro = None

    def enrutar(self, x):
        # El router trabaja en fp32 aunque el resto use bf16, como la «precisión selectiva» de Switch.
        entrada = x.float()
        if self.training and self.ruido > 0:
            entrada = entrada * torch.empty_like(entrada).uniform_(1 - self.ruido, 1 + self.ruido)
        with torch.autocast(x.device.type, enabled=False):
            return self.router(entrada)

    def forward(self, x):
        logits = self.enrutar(x)
        probs_top, indices = F.softmax(logits, dim=-1).topk(self.top_k, dim=-1)
        pesos = (probs_top / probs_top.sum(dim=-1, keepdim=True)).to(x.dtype)
        if self.training:
            self.registro = (logits, indices)

        salida = torch.zeros_like(x)
        for e, experto in enumerate(self.expertos):
            lote, token, rango = torch.where(indices == e)
            if lote.numel():
                aporte = pesos[lote, token, rango].unsqueeze(-1) * experto(x[lote, token]).to(x.dtype)
                salida[lote, token] = salida[lote, token] + aporte
        return salida

    def perdidas(self, mascara):
        """z-loss y balance de carga de la última pasada, sin contar el padding."""
        logits, indices = self.registro
        self.registro = None
        mascara = mascara.to(logits.dtype).unsqueeze(-1)
        tokens = mascara.sum(dim=1).clamp_min(1)

        z = (torch.logsumexp(logits, dim=-1).pow(2) * mascara.squeeze(-1)).sum() / tokens.sum()
        despacho = F.one_hot(indices, self.num_expertos).sum(dim=-2).to(logits.dtype)
        fraccion = (despacho * mascara).sum(dim=1) / (tokens * self.top_k)
        prob_media = (F.softmax(logits, dim=-1) * mascara).sum(dim=1) / tokens
        balance = (self.num_expertos * (fraccion * prob_media).sum(dim=-1)).mean()  # vale 1 si la carga es pareja
        return z, balance


class MT5MoEForConditionalGeneration(MT5ForConditionalGeneration):
    config_class = MT5MoEConfig

    def __init__(self, config):
        super().__init__(config)
        paso = config.moe_paso_disperso
        for pila, es_decoder in ((self.encoder, False), (self.decoder, True)):
            for i, bloque in enumerate(pila.block):
                if paso == 1 or i % paso == 1:
                    bloque.layer[-1].DenseReluDense = CapaMoE(config, es_decoder)

    def capas_moe(self):
        return [m for m in self.modules() if isinstance(m, CapaMoE)]

    def perdidas_router(self, mascara_encoder, mascara_decoder):
        """Promedia por capa dentro de cada pila y suma encoder y decoder, como Switch."""
        por_pila = {False: [], True: []}
        for capa in self.capas_moe():
            por_pila[capa.es_decoder].append(capa.perdidas(mascara_decoder if capa.es_decoder else mascara_encoder))
        z = sum(torch.stack([p[0] for p in v]).mean() for v in por_pila.values())
        balance = sum(torch.stack([p[1] for p in v]).mean() for v in por_pila.values())
        return z, balance

## 6. Upcycling desde `google/mt5-small`

Cada capa dispersa recibe cuatro copias de la feed-forward densa. El resto de los pesos se copia sin cambios y solo el enrutador parte aleatorio. La celda se detiene si el MoE no reproduce las salidas del modelo denso.

In [ ]:
def upcycling(denso):
    datos = {k: v for k, v in denso.config.to_dict().items() if k not in ("model_type", "architectures", "transformers_version")}
    config = MT5MoEConfig(**datos, moe_expertos=NUM_EXPERTOS, moe_top_k=TOP_K,
                          moe_paso_disperso=PASO_DISPERSO, moe_ruido_router=RUIDO_ROUTER)

    # from_pretrained conserva lm_head separado de los embeddings, como viene en el checkpoint de mT5.
    moe, info = MT5MoEForConditionalGeneration.from_pretrained(MODELO, config=config, output_loading_info=True)
    faltan = [k for k in info["missing_keys"] if ".DenseReluDense.expertos." not in k and ".DenseReluDense.router." not in k]
    sobran = [k for k in info["unexpected_keys"] if ".DenseReluDense." not in k]
    if faltan or sobran:
        raise RuntimeError(f"Pesos incompatibles. Faltan: {faltan[:3]} | Sobran: {sobran[:3]}")

    for pila_moe, pila_densa in ((moe.encoder, denso.encoder), (moe.decoder, denso.decoder)):
        for bloque_moe, bloque_denso in zip(pila_moe.block, pila_densa.block):
            capa = bloque_moe.layer[-1].DenseReluDense
            if isinstance(capa, CapaMoE):
                pesos_densos = bloque_denso.layer[-1].DenseReluDense.state_dict()
                for experto in capa.expertos:
                    experto.load_state_dict({k: v.clone() for k, v in pesos_densos.items()}, assign=True)
                capa.router.weight = nn.Parameter(torch.empty(capa.router.weight.shape).normal_(mean=0.0, std=0.01))
    return moe


tok = AutoTokenizer.from_pretrained(MODELO)
denso = AutoModelForSeq2SeqLM.from_pretrained(MODELO).eval()
c = denso.config
if (c.d_model, c.d_ff, c.num_layers, c.num_decoder_layers, c.feed_forward_proj) != (512, 1024, 8, 8, "gated-gelu"):
    raise RuntimeError("El modelo base no tiene la arquitectura esperada de mT5-small.")

mod = upcycling(denso).eval()

muestra = val[:LOTE]
enc = tok([x[0] for x in muestra], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
lab = tok(text_target=[x[1] for x in muestra], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).input_ids
with torch.no_grad():
    logits_denso = denso(**enc, labels=lab).logits
    diferencia = ((logits_denso - mod(**enc, labels=lab).logits).abs().max() / logits_denso.abs().max()).item()
# Tolera solo el redondeo de fp32: los estados ocultos de mT5 tienen valores muy grandes.
if diferencia > 1e-3:
    raise RuntimeError(f"El MoE no reproduce al modelo denso (diferencia relativa {diferencia:.2e}).")

n_denso = sum(p.numel() for p in denso.parameters())
n_moe = sum(p.numel() for p in mod.parameters())
print(f"Capas dispersas: {len(mod.capas_moe())} | parámetros: {n_denso / 1e6:.1f} M → {n_moe / 1e6:.1f} M")
print(f"Upcycling verificado: diferencia relativa con el denso {diferencia:.2e}")

del denso
mod = mod.to(dev)

## 7. Entrenar

In [ ]:
opt = Adafactor(mod.parameters(), lr=LR, relative_step=False, scale_parameter=False, warmup_init=False)
mod.config.use_cache = False


def codificar(lote):
    enc = tok([x[0] for x in lote], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(dev)
    lab = tok(text_target=[x[1] for x in lote], return_tensors="pt", padding=True, truncation=True,
              max_length=MAX_LENGTH).input_ids.to(dev)
    lab[lab == tok.pad_token_id] = -100
    return enc, lab


def guardar_mejor():
    mod.config.use_cache = True
    mod.save_pretrained(DIR_LOCAL)
    tok.save_pretrained(DIR_LOCAL)
    mod.config.use_cache = False


shutil.rmtree(DIR_LOCAL, ignore_errors=True)
mejor, mejor_epoca, historial = float("inf"), None, []
for ep in range(1, EPOCAS + 1):
    mod.train()
    random.shuffle(train)
    total_ce = total_z = total_balance = 0.0
    n_pasos = math.ceil(len(train) / LOTE)
    for i in range(0, len(train), LOTE):
        enc, lab = codificar(train[i:i + LOTE])
        with torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
            ce = mod(**enc, labels=lab).loss
            z, balance = mod.perdidas_router(enc.attention_mask, lab != -100)
            loss = ce + COEF_Z * z + COEF_BALANCE * balance
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        total_ce += ce.item()
        total_z += z.item()
        total_balance += balance.item()
        if (i // LOTE) % 100 == 0:
            print(f"  época {ep} paso {i // LOTE}/{len(train) // LOTE} ce {ce.item():.3f} z {z.item():.3f} "
                  f"balance {balance.item():.3f}", flush=True)

    mod.eval()
    perd = 0.0
    with torch.no_grad():
        for i in range(0, len(val), LOTE):
            b = val[i:i + LOTE]
            enc, lab = codificar(b)
            with torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
                perd += mod(**enc, labels=lab).loss.item() * len(b)
    perd /= len(val)
    fila = {"epoca": ep, "perdida_entrenamiento": total_ce / n_pasos, "perdida_validacion": perd,
            "z_loss": total_z / n_pasos, "balance": total_balance / n_pasos}
    historial.append(fila)
    print(f"época {ep}: entrenamiento {fila['perdida_entrenamiento']:.3f} | validación {perd:.3f} | "
          f"z {fila['z_loss']:.3f} | balance {fila['balance']:.3f}", flush=True)
    if perd < mejor:  # se compara solo la entropía cruzada, igual que en el denso
        mejor, mejor_epoca = perd, ep
        guardar_mejor()

if mejor_epoca is None:
    raise RuntimeError("La pérdida de validación no fue válida en ninguna época; no se guardó el modelo.")
print(f"Mejor época: {mejor_epoca} (validación {mejor:.3f})")

## 8. Guardar los pesos en Drive

In [ ]:
with open(f"{DIR_LOCAL}/entrenamiento.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": MODELO, "precision": PRECISION, "epocas": EPOCAS, "lote": LOTE, "lr": LR, "val": VAL,
               "semilla": SEMILLA, "max_length": MAX_LENGTH, "num_expertos": NUM_EXPERTOS, "top_k": TOP_K,
               "paso_disperso": PASO_DISPERSO, "coef_balance": COEF_BALANCE, "coef_z": COEF_Z,
               "ruido_router": RUIDO_ROUTER, "mejor_epoca": mejor_epoca, "historial": historial},
              f, ensure_ascii=False, indent=2)

temporal = f"{DIR_PESOS}.tmp"
shutil.rmtree(temporal, ignore_errors=True)
shutil.copytree(DIR_LOCAL, temporal)
if os.path.isdir(DIR_PESOS):
    shutil.rmtree(DIR_PESOS)
os.rename(temporal, DIR_PESOS)

pesos = f"{DIR_PESOS}/model.safetensors"
if os.path.getsize(pesos) != os.path.getsize(f"{DIR_LOCAL}/model.safetensors"):
    raise RuntimeError("La copia de los pesos en Drive quedó incompleta.")
print(f"Pesos guardados en {DIR_PESOS} ({os.path.getsize(pesos) / 1024**3:.2f} GB)")

## 9. Evaluar el modelo guardado

Para cargar estos pesos en otro notebook hay que ejecutar antes la celda del paso 5 y usar `MT5MoEForConditionalGeneration.from_pretrained(ruta)`.

In [ ]:
def traducir(modelo, frases, lote=16):
    salidas = []
    modelo.eval()
    for i in range(0, len(frases), lote):
        enc = tok(frases[i:i + lote], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(dev)
        with torch.no_grad(), torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
            out = modelo.generate(**enc, max_new_tokens=96, num_beams=4)
        salidas += tok.batch_decode(out, skip_special_tokens=True)
    return salidas


del mod, opt
torch.cuda.empty_cache()
tok = AutoTokenizer.from_pretrained(DIR_PESOS)
modelo_final, info = MT5MoEForConditionalGeneration.from_pretrained(DIR_PESOS, output_loading_info=True)
if info["missing_keys"] or info["unexpected_keys"]:
    raise RuntimeError(f"Pesos incompletos: {list(info['missing_keys'])[:3]} {list(info['unexpected_keys'])[:3]}")
modelo_final = modelo_final.to(dev)
modelo_final.config.use_cache = True

filas, salidas = [], []
for ruta, ps in conjuntos.items():
    hip = traducir(modelo_final, [p["chileno"] for p in ps])
    ref = [p["neutro"] for p in ps]
    salidas += [(os.path.basename(ruta), p["chileno"], r, p["termino"], h) for p, r, h in zip(ps, ref, hip)]
    bleu = sacrebleu.corpus_bleu(hip, [ref]).score
    chrf = sacrebleu.corpus_chrf(hip, [ref]).score
    print(f"{os.path.basename(ruta):<24} {NOMBRE:<18} BLEU {bleu:5.1f} chrF {chrf:5.1f}")
    filas.append({"sistema": NOMBRE, "escenario": "", "bleu": f"{bleu:.1f}", "chrf": f"{chrf:.1f}", "n_pares": len(ps),
                  "pares": ruta, "nota": f"{MODELO} + MoE {NUM_EXPERTOS} expertos top-{TOP_K}; {PRECISION}; ajustado con borradores del LLM"})

with open(RUTA_PREDICCIONES, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["conjunto", "chileno", "referencia", "termino", NOMBRE])
    w.writerows(salidas)
guardar_resumen(RUTA_RESUMEN, filas)

## 10. Confirmar la escritura en Drive

In [ ]:
print("Pesos:", DIR_PESOS)
print("Predicciones:", RUTA_PREDICCIONES)
print("Resumen:", RUTA_RESUMEN)
drive.flush_and_unmount()
print("Drive sincronizado y desmontado.")